# AH | Diagnósticos analíticos

**Autor:** Alan Arturo Hernández Gutiérrez.

Contrasta fechas, importes, categorías, canales y cobertura de atributos. Sus resultados agregados se guardan en evidencia/.

Este cuaderno define las operaciones que utilizan los cuadernos de `notebooks/`. Ejecute sus celdas en orden. El flujo completo se inicia desde 01 o 02.

## Raíz del proyecto

La ubicación se resuelve desde la raíz, `src/` o `notebooks/`, sin guardar rutas personales.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name.lower() in {'src', 'notebooks'}:
    ROOT = ROOT.parent
if not (ROOT / 'config' / 'proyecto.json').is_file():
    raise RuntimeError('Inicie Jupyter desde la raíz del proyecto, src/ o notebooks/.')


## Bibliotecas y parámetros

Se preparan las bibliotecas y constantes utilizadas por las operaciones siguientes.

In [ ]:
from pathlib import Path

from io import BytesIO

import zipfile, json, sys, os

import pandas as pd

import numpy as np


## Serializar diagnósticos

Convierte escalares de NumPy y nulos de pandas a valores compatibles con el informe JSON.

In [ ]:
def json_default(value):
    if value is pd.NA:
        return None
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(type(value).__name__)


## Bibliotecas y parámetros

Se preparan las bibliotecas y constantes utilizadas por las operaciones siguientes.

In [ ]:
out = ROOT / "evidencia"

out.mkdir(exist_ok=True)


## Leer las fuentes para contraste

Carga los siete CSV como texto para revisar conjuntamente importes, fechas, clientes y categorías.

In [ ]:
def _leer_fuentes_diagnostico(archive):
    with zipfile.ZipFile(archive) as z:

        def read(name):
            return pd.read_csv(BytesIO(z.read(name + ".csv")), dtype="string")

        f = read("FAC_VENTAS")
        c = read("DIM_CLIENTES")
        s = read("DIM_TIENDA")
        g = read("DIM_MUNICIPIO")
        seg = read("DIM_SEGMENTOS")
        p = read("DIM_PRODUCTO")
        cat = read("DIM_CATEGORIA")
    return (f, c, s, g, seg, p, cat)


## Consolidar actividad y tickets

Prepara las variables numéricas y de fecha; une canal e identidad y resume atributos por cliente y montos por ticket.

In [ ]:
def _actividad_diagnostico(f, c, s):
    f["MONTO"] = pd.to_numeric(f.MONTO)
    f["ITEMS"] = pd.to_numeric(f.ITEMS)
    f["date"] = pd.to_datetime(f.FECHA, format="%Y-%m-%d")
    tx = f.merge(
        c[["CUENTA_CVE", "CLIENTE_CVE"]], on="CUENTA_CVE", how="left", validate="many_to_one"
    ).merge(s[["TIENDA_CVE", "TIPO"]], on="TIENDA_CVE", how="left", validate="many_to_one")
    active = c[c.CLIENTE_CVE.isin(tx.CLIENTE_CVE.unique())]
    ac = active.groupby("CLIENTE_CVE").agg(
        age=("EDAD", "first"),
        age_versions=("EDAD", "nunique"),
        municipality=("MUNICIPIO_CVE", "first"),
        accounts=("CUENTA_CVE", "nunique"),
    )
    ticket = f.groupby("TICKET").agg(amount=("MONTO", "sum"), rows=("MONTO", "size"))
    return (tx, ac, ticket)


## Revisar importes negativos por categoría

Une las categorías y calcula líneas, importes firmados y negativos dentro de cada categoría.

In [ ]:
def _categorias_diagnostico(tx, p, cat):
    bycat = (
        tx[["SKU_CVE", "MONTO"]]
        .merge(p[["SKU_CVE", "CATEGORIA_CVE"]], on="SKU_CVE", how="left", validate="many_to_one")
        .merge(cat, on="CATEGORIA_CVE", how="left", validate="many_to_one")
    )
    bc = (
        bycat.assign(
            negative=bycat.MONTO.lt(0), negative_amount=bycat.MONTO.where(bycat.MONTO.lt(0), 0)
        )
        .groupby("CATEGORIA_NOMBRE", dropna=False)
        .agg(
            rows=("MONTO", "size"),
            negative_rows=("negative", "sum"),
            signed_amount=("MONTO", "sum"),
            negative_amount=("negative_amount", "sum"),
        )
        .reset_index()
    )
    return bc


## Contrastar cuatro reglas de lectura

Compara los universos y los canales iniciales al conservar o excluir importes negativos y servicios.

In [ ]:
def _sensibilidades_diagnostico(tx, p, cat):
    scenarios = {}
    category_map = p.merge(cat, on="CATEGORIA_CVE", how="left", validate="many_to_one").set_index(
        "SKU_CVE"
    )["CATEGORIA_NOMBRE"]
    tx["category"] = tx.SKU_CVE.map(category_map)
    non_services = tx.category.notna() & tx.category.ne("SERVICIOS").fillna(False)
    for name, q in [
        ("all_operation_1", tx),
        ("positive_amount_only", tx.loc[tx.MONTO.gt(0)]),
        ("excluding_services", tx.loc[non_services]),
        ("positive_excluding_services", tx.loc[tx.MONTO.gt(0) & non_services]),
    ]:
        first = q.groupby("CLIENTE_CVE").date.min()
        fr = q[q.date.eq(q.CLIENTE_CVE.map(first))]
        flags = (
            fr.assign(
                online=fr.TIPO.eq("Internet"), physical=fr.TIPO.eq("Fisica"), unknown=fr.TIPO.isna()
            )
            .groupby("CLIENTE_CVE")[["online", "physical", "unknown"]]
            .max()
        )
        scenarios[name] = {
            "customers": len(first),
            "first_day_online_only": int((flags.online & ~flags.physical & ~flags.unknown).sum()),
            "first_day_online_physical": int((flags.online & flags.physical).sum()),
            "first_day_has_unknown": int(flags.unknown.sum()),
            "ever_online": int(q.loc[q.TIPO.eq("Internet"), "CLIENTE_CVE"].nunique()),
        }
    return scenarios


## Organizar la evidencia de diagnóstico

Reúne cobertura de atributos, integridad territorial, calendario, marcas e importes en un resultado agregado.

In [ ]:
def _reporte_diagnosticos(f, c, s, g, seg, p, ac, ticket, bc, scenarios):
    report = {
        "negative_amounts": {
            "rows": int(f.MONTO.lt(0).sum()),
            "share": float(f.MONTO.lt(0).mean()),
            "negative_sum": float(f.loc[f.MONTO.lt(0), "MONTO"].sum()),
            "positive_sum": float(f.loc[f.MONTO.gt(0), "MONTO"].sum()),
            "zero_rows": int(f.MONTO.eq(0).sum()),
            "negative_ticket_totals": int(ticket.amount.lt(0).sum()),
            "zero_ticket_totals": int(ticket.amount.eq(0).sum()),
            "all_operations_labeled_1": bool(f.TIPO_OPERACION.eq("1").all()),
            "by_category": bc.to_dict("records"),
        },
        "active_customers": {
            "customers": len(ac),
            "age_missing": int(ac.age.isna().sum()),
            "age_conflicts": int(ac.age_versions.gt(1).sum()),
            "home_missing": int(ac.municipality.isna().sum()),
            "home_nonnull_not_in_catalog": int(
                (ac.municipality.notna() & ~ac.municipality.isin(g.MUNICIPIO_CVE)).sum()
            ),
            "segment_missing": int((~ac.index.isin(seg.CLIENTE_CVE)).sum()),
            "accounts_more_than_one": int(ac.accounts.gt(1).sum()),
            "max_accounts": int(ac.accounts.max()),
        },
        "geography": {
            "all_keys_five_digits": bool(g.MUNICIPIO_CVE.str.fullmatch("\\d{5}").all()),
            "state_prefix_matches": int(g.MUNICIPIO_CVE.str[:2].eq(g.ESTADO_CLAVE).sum()),
            "invalid_client_municipality_values": list(
                c.loc[
                    c.MUNICIPIO_CVE.notna() & ~c.MUNICIPIO_CVE.isin(g.MUNICIPIO_CVE),
                    "MUNICIPIO_CVE",
                ].unique()
            ),
            "invalid_store_municipality_values": list(
                s.loc[
                    s.MUNICIPIO_CVE.notna() & ~s.MUNICIPIO_CVE.isin(g.MUNICIPIO_CVE),
                    "MUNICIPIO_CVE",
                ].unique()
            ),
        },
        "dates": {
            "calendar_days": int((f.date.max() - f.date.min()).days + 1),
            "days_observed": int(f.date.nunique()),
            "days_without_rows": int((f.date.max() - f.date.min()).days + 1 - f.date.nunique()),
            "format": "YYYY-MM-DD",
        },
        "brands": {
            "brand_codes": int(p.MARCA_CVE.nunique()),
            "brand_codes_used_in_multiple_categories": int(
                p.groupby("MARCA_CVE").CATEGORIA_CVE.nunique().gt(1).sum()
            ),
        },
        "amount_rule_sensitivity": scenarios,
    }
    return report


## Diagnosticar fuentes

Calcula las comprobaciones adicionales de fechas, atributos, geografía y sensibilidad.

In [ ]:
def diagnosticar_fuentes(archive):
    archive = Path(archive)
    f, c, s, g, seg, p, cat = _leer_fuentes_diagnostico(archive)
    tx, ac, ticket = _actividad_diagnostico(f, c, s)
    bc = _categorias_diagnostico(tx, p, cat)
    scenarios = _sensibilidades_diagnostico(tx, p, cat)
    report = _reporte_diagnosticos(f, c, s, g, seg, p, ac, ticket, bc, scenarios)
    (out / "diagnosticos_adicionales.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2, default=json_default), encoding="utf-8"
    )
    return report
